# Sesión 14 sep 2026 — Pandas y procesamiento de datos

| Recurso | Fichero |
| --- | --- |
| Ejercicio de clase | [`ejercicios/E1_pandas.md`](ejercicios/E1_pandas.md) |
| Dataset principal | [`Datos/ventas.csv`](Datos/ventas.csv) |
| Dataset auxiliar | [`Datos/iris.csv`](Datos/iris.csv) |
| Documentación | [pandas.pydata.org/docs](https://pandas.pydata.org/docs/) |

Este notebook es la **referencia amplia** de la sesión 2: conceptos, demos ejecutables, flujos recomendados, anti-patrones y puente al ejercicio E1 / Proyecto I. Úsalo en clase y como manual cuando proceses datos en el resto del curso.

**Cómo trabajarlo:** ejecuta las celdas en orden con el venv del curso activado (`pandas` instalado).


---

## 0. Objetivos de aprendizaje

Al terminar esta sesión (y este notebook) deberías ser capaz de:

1. Explicar por qué pandas es la herramienta base de **ingesta y transformación** en DSIA.
2. Distinguir `Series` y `DataFrame` y crearlos de forma controlada.
3. Cargar CSV con **`pathlib`** (sin rutas absolutas del autor).
4. Diagnosticar un dataset: `shape`, `dtypes`, nulos, valores sospechosos.
5. Filtrar, seleccionar (`loc`/`iloc`), agrupar y agregar.
6. Crear columnas derivadas con operaciones vectoriales (evitar `apply` innecesario).
7. Separar filas **válidas** e **inválidas** con reglas explícitas.
8. Exportar resultados (`CSV` / resumen JSON) listos para la siguiente etapa del pipeline.

---


## 1. Por qué esta sesión importa en DSIA

### 1.1 Qué entregamos

En DSIA no entregamos “un Excel retocado a mano”. Entregamos un **flujo reproducible**:

```text
datos crudos → diagnóstico → limpieza/validación → agregaciones → artefacto (CSV/JSON) → (más adelante) API / IA
```

Si la limpieza vive solo en celdas sueltas sin criterios claros, el Proyecto I y el E2E se rompen.

### 1.2 Mapa mental del semestre

```text
Sesión 1  →  venv + Git
Sesión 2  →  pandas (hoy)
Sesión 3  →  arquitectura / Clean Code / SOLID
Después   →  tests, APIs IA, E2E, deploy
```

La sesión 3 **refactorizará** lo que hoy exploramos aquí hacia un paquete (`loader` / `validator` / `metrics`).

### 1.3 Analogías útiles

| Concepto | Analogía |
| --- | --- |
| `Series` | Una columna etiquetada |
| `DataFrame` | Tabla (colecciones de Series alineadas) |
| `dtype` | El “tipo de casilla” de la hoja |
| filtro booleano | Quedarse con las filas que cumplen una regla |
| `groupby` | Tablas dinámicas de Excel, pero en código |
| validación | Control de calidad antes de fabricar KPIs |
| `pathlib` | GPS relativo al proyecto, no “en mi Escritorio” |

### 1.4 Regla de oro

> **Explora en el notebook; formula reglas explícitas; deja el resultado exportable.**

---


## 2. Entorno e imports

Comprueba que el intérprete del notebook es el del venv del curso.


In [ ]:
from __future__ import annotations

from datetime import date
from pathlib import Path
import json

import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("cwd idea:", Path.cwd())
DATA_DIR = Path("Datos")  # ejecuta el notebook desde 1_programacion_avanzada_python/
print("Datos existe:", DATA_DIR.exists())


Si `Datos existe: False`, cambia el directorio de trabajo del notebook a la carpeta `1_programacion_avanzada_python/` (o ajusta `DATA_DIR`).

---

## 3. `Series`: la columna tipada

Una **Series** es una estructura **unidimensional** con índice. Documentación: [`pandas.Series`](https://pandas.pydata.org/docs/reference/api/pandas.Series.html).

### 3.1 Creación básica


In [ ]:
serie = pd.Series(["a", "b", "c"])
serie


### 3.2 Series con índice semántico (p. ej. fechas)


In [ ]:
ts = pd.Series(
    {
        date(2026, 9, 7): 10,
        date(2026, 9, 14): 15,
        date(2026, 9, 21): 15,
    }
)
ts


**Para DSIA:** casi siempre trabajarás con columnas dentro de un `DataFrame`. La Series aparece al hacer `df["columna"]` o al agregar.

---

## 4. `DataFrame`: la tabla de trabajo

Un **DataFrame** es una estructura **bidimensional** (filas × columnas). Cada columna es una Series. Documentación: [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).

### 4.1 Crear desde filas + nombres de columnas


In [ ]:
df_demo = pd.DataFrame(
    data=[
        [date(2026, 9, 7), 1, "a"],
        [date(2026, 9, 14), 2, "b"],
        [date(2026, 9, 21), 3, "c"],
    ],
    columns=["timestamp", "integers", "strings"],
)
df_demo


### 4.2 Índice: cuidado con `inplace`

`set_index` **devuelve una copia** salvo que uses `inplace=True` (preferible asignar).


In [ ]:
df_indexed = df_demo.set_index("timestamp")
df_indexed


### 4.3 Crear columna a columna


In [ ]:
df2 = pd.DataFrame()
df2["timestamp"] = [date(2026, 9, 7), date(2026, 9, 14), date(2026, 9, 21)]
df2["integers"] = [1, 2, 3]
df2["strings"] = ["a", "b", "c"]
df2


In [ ]:
# Comparación elemento a elemento (útil en tests / checks)
df_demo.reset_index(drop=True) == df2


### 4.4 Paso a NumPy

`DataFrame.to_numpy()` (preferible a `.values` en código nuevo).


In [ ]:
df_demo.to_numpy()


---

## 5. Lectura de datos (I/O en el borde)

### 5.1 Rutas: absoluta vs relativa

| Tipo | Ejemplo | Problema |
| --- | --- | --- |
| Absoluta | `/Users/yo/Desktop/ventas.csv` | No funciona en otra máquina |
| Relativa al repo | `Datos/ventas.csv` | Portable si fijas el cwd |

En DSIA usamos **`pathlib.Path`** relativo al proyecto.

### 5.2 CSV (formato principal del curso)

Documentación: [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).


In [ ]:
ventas_path = DATA_DIR / "ventas.csv"
ventas = pd.read_csv(ventas_path)
ventas


### 5.3 Dataset auxiliar: iris

Útil para practicar `groupby` / filtros con más filas. Mismo patrón de ruta.


In [ ]:
iris_path = DATA_DIR / "iris.csv"
iris = pd.read_csv(iris_path)
iris.head()


### 5.4 Otros formatos (idea; no obligatorios hoy)

| Formato | Función | Nota |
| --- | --- | --- |
| JSON | `pd.read_json` | Común en APIs |
| Excel | `pd.read_excel` | Requiere motor (`openpyxl` / `calamine`) |
| Parquet | `pd.read_parquet` | Ideal en pipelines mayores |

**No hace falta** Excel/JSON en el ejercicio E1; el contrato del curso es CSV.

---

## 6. Diagnóstico de un DataFrame (antes de limpiar)

Orden recomendado siempre:

```text
cargar → shape/dtypes → head → na → describe/value_counts → sospechosos
```

### 6.1 Forma, tipos y nulos


In [ ]:
print("shape:", ventas.shape)
print()
print("dtypes:")
print(ventas.dtypes)
print()
print("nulos por columna:")
print(ventas.isna().sum())


### 6.2 Vista rápida y estadísticas


In [ ]:
ventas.head()


In [ ]:
ventas.describe(include="all")


In [ ]:
ventas["region"].value_counts()


### 6.3 Lectura crítica (hazla por escrito en E1)

Con `ventas.csv` del curso deberías detectar al menos:

- una fila con `unidades` nula,
- una fila con `precio_unitario` negativo (o no positivo).

Eso alimenta las **reglas de validación**, no un “borrado a ojo”.

---

## 7. Selección, filtros y localización

### 7.1 Filtro booleano


In [ ]:
# Ejemplo sobre iris: filas por encima de la media de sepal.length
# (normalizamos nombres de columnas con puntos → guiones bajos)
iris = iris.rename(columns=lambda c: c.replace(".", "_"))
iris[iris["sepal_length"] >= iris["sepal_length"].mean()].head()


### 7.2 `loc` (etiquetas) vs `iloc` (posición)


In [ ]:
# loc: etiqueta de fila + nombre de columna
iris.loc[0, "sepal_length"]


In [ ]:
# iloc: posición entera
iris.iloc[19]


**Heurística:** usa nombres de columna (`loc` / `[]`) en pipelines; `iloc` solo cuando la posición importa de verdad.

---

## 8. Agrupaciones y agregaciones

Documentación útil: [`DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html).

### 8.1 Media por categoría


In [ ]:
iris.groupby("variety")["sepal_length"].mean()


### 8.2 Varias métricas con `agg`


In [ ]:
iris.groupby("variety").agg(
    petal_length_median=("petal_length", "median"),
    petal_width_max=("petal_width", "max"),
)


---

## 9. Operaciones vectoriales (preferibles a bucles)

### 9.1 Aritmética columna a columna


In [ ]:
(iris["sepal_length"] / iris["petal_length"]).head()


### 9.2 `map` / `apply`: úsalos con criterio

- **Vectorizado** (`+`, máscaras, `np.where`) → rápido y claro.
- `Series.map` → transformaciones elemento a elemento simples.
- `DataFrame.apply(..., axis=1)` → flexible pero **lento**; en datos grandes, último recurso.


In [ ]:
iris["petal_width"].map(lambda x: int(x) if x > 1.5 else x).head()


In [ ]:
iris.apply(lambda row: 5 * row["petal_length"] + 6 * row["petal_width"], axis=1).head()


Equivalente más idiomático (vectorizado):


In [ ]:
(5 * iris["petal_length"] + 6 * iris["petal_width"]).head()


---

## 10. Limpieza básica: tipos, duplicados, columnas

### 10.1 Cambiar tipo


In [ ]:
iris["sepal_width"].astype(str).iloc[0]


### 10.2 Duplicados y borrado de columnas (sin romper el original)


In [ ]:
print("duplicados:", iris.duplicated().sum())
iris_no_dupes = iris.drop_duplicates()
iris_without_width = iris.drop(columns=["sepal_width"])
iris_without_width.head()


**Hábitos:**

- Preferir `drop(columns=[...])` a `drop(..., axis=1)` por legibilidad.
- Evitar `inplace=True` en material de curso (más difícil de razonar).

---

## 11. Mini-pipeline DSIA con `ventas.csv` (núcleo del E1)

Este es el patrón que reutilizarás en arquitectura (21 sep):

```text
cargar → validar → agregar → exportar
```

### 11.1 Cargar con pathlib


In [ ]:
def cargar_ventas(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No existe el fichero: {path}")
    return pd.read_csv(path)


raw = cargar_ventas(DATA_DIR / "ventas.csv")
raw


### 11.2 Validar: separar válidos y errores

Reglas mínimas del ejercicio E1:

- `unidades` numérica y `> 0`
- `precio_unitario` numérico y `> 0`
- `importe = unidades * precio_unitario` solo en válidos

**Checkpoint:** con el CSV del curso → **140 válidas** y **10 inválidas**.


In [ ]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    work = frame.copy()
    work["unidades"] = pd.to_numeric(work["unidades"], errors="coerce")
    work["precio_unitario"] = pd.to_numeric(work["precio_unitario"], errors="coerce")

    ok = (
        work["unidades"].notna()
        & (work["unidades"] > 0)
        & work["precio_unitario"].notna()
        & (work["precio_unitario"] > 0)
    )

    validos = work.loc[ok].copy()
    errores = work.loc[~ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores


validos, errores = validar_ventas(raw)
print(f"válidas: {len(validos)} | inválidas: {len(errores)}")
errores


### 11.3 Agregaciones de negocio


In [ ]:
# Importe total por región (desc)
importe_por_region = (
    validos.groupby("region", as_index=False)["importe"]
    .sum()
    .sort_values("importe", ascending=False)
)
importe_por_region


In [ ]:
# Top 3 productos por importe
top_productos = (
    validos.groupby("producto", as_index=False)["importe"]
    .sum()
    .sort_values("importe", ascending=False)
    .head(3)
)
top_productos


In [ ]:
# cliente_id con más de una compra
compras_por_cliente = validos["cliente_id"].value_counts()
clientes_recurrentes = compras_por_cliente[compras_por_cliente > 1]
clientes_recurrentes


### 11.4 Exportar artefactos


In [ ]:
salida_csv = DATA_DIR / "ventas_limpias.csv"
salida_json = DATA_DIR / "calidad_datos.json"

validos.to_csv(salida_csv, index=False)

calidad = {
    "filas_totales": int(len(raw)),
    "filas_validas": int(len(validos)),
    "filas_invalidas": int(len(errores)),
    "importe_total": float(validos["importe"].sum()),
}
salida_json.write_text(json.dumps(calidad, indent=2, ensure_ascii=False), encoding="utf-8")

print("escrito:", salida_csv)
print("escrito:", salida_json)
calidad


> **Nota:** `ventas_limpias.csv` y `calidad_datos.json` son salidas generadas. No hace falta versionarlas si tu `.gitignore` las excluye; sí debes saber regenerarlas.

---

## 12. Anti-patrones frecuentes (lista negra)

1. Rutas absolutas (`/Users/.../ventas.csv`).
2. Limpiar “a ojo” en celdas sin función reutilizable.
3. Sobrescribir el DataFrame crudo sin `copy()` y perder trazabilidad.
4. `for row in df.iterrows()` para todo (lento y verboso).
5. `apply(axis=1)` cuando basta una operación vectorial.
6. Tratar nulos como ceros **sin criterio** de negocio.
7. Agregar KPIs **antes** de validar.
8. Encadenar 30 transformaciones en una sola celda sin nombres.
9. Dependender del orden mágico de ejecución del notebook (“Run All” frágil).
10. Dejar el resultado solo en pantalla: sin CSV/JSON no hay pipeline.

---

## 13. De notebook a paquete (adelanto)

Hoy está bien explorar aquí. Mañana (arquitectura) querrás:

```text
ventas_app/
  loader.py      ← cargar_ventas
  validator.py   ← validar_ventas
  metrics.py     ← groupby / tops
  cli.py         ← orquestación
```

Misma lógica; mejor frontera.

---

## 14. Autoevaluación

1. ¿Qué diferencia una Series de un DataFrame?
2. ¿Por qué `pathlib` frente a rutas absolutas?
3. ¿Qué imprime tu diagnóstico antes de limpiar?
4. ¿Por qué validar antes de `groupby`?
5. ¿Cuándo evitarías `apply(axis=1)`?
6. ¿Qué debe contener `calidad_datos.json` en el E1?

Si dudas en más de dos, rehaz la sección 11 y el ejercicio [`E1_pandas.md`](ejercicios/E1_pandas.md).

---

## 15. Checklist de salida

### Código

- [ ] Notebook ejecuta desde `1_programacion_avanzada_python/`
- [ ] `ventas.csv` cargado con `Path`
- [ ] Diagnóstico: `shape`, `dtypes`, nulos
- [ ] `validar_ventas` → 140 válidas / 10 inválidas
- [ ] Agregados: región, top productos, clientes recurrentes
- [ ] Exportados `ventas_limpias.csv` y `calidad_datos.json`

### Comprensión

- [ ] Sé explicar el flujo cargar → validar → agregar → exportar
- [ ] Sé 3 anti-patrones que no cometeré en el Proyecto I
- [ ] Sé qué moveré a módulos en la sesión de arquitectura

---

## 16. Para la sesión del 21 sep

1. Deja el E1 resuelto en tu repo del Proyecto I.
2. No borres las funciones `cargar` / `validar` / `agregar`: las reubicarás.
3. Ojea `03_arquitectura_patrones.md` y el demo OOP/SOLID.
4. Trae preguntas concretas (“¿el importe va en validator o en metrics?”).

---

## 17. Apéndice A — Chuleta rápida

```python
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path("Datos/ventas.csv"))
df.shape, df.dtypes, df.isna().sum()
df.head(); df.describe(include="all")
df[df["region"] == "Norte"]
df.loc[0, "producto"]; df.iloc[0]
df.groupby("region")["unidades"].sum()
df.assign(importe=df["unidades"] * df["precio_unitario"])
df.to_csv("Datos/salida.csv", index=False)
```

---

## 18. Apéndice B — Glosario corto EN/ES

| EN | ES / nota |
| --- | --- |
| DataFrame | tabla / marco de datos |
| Series | serie / columna etiquetada |
| dtype | tipo de dato de columna |
| missing / NA | valor ausente / nulo |
| filter / mask | filtro / máscara booleana |
| groupby | agrupación |
| aggregate (`agg`) | agregar / resumir |
| vectorized | vectorizado (sin bucle explícito) |
| schema | esquema de columnas / contrato |
| tidy data | datos ordenados (filas = observaciones) |

---

## 19. Apéndice C — Preguntas típicas de clase

**¿Puedo hacer el E1 solo en Excel y pegar el CSV?**  
No para la entrega de la asignatura: el valor está en el **código reproducible**.

**¿Obligatorio usar funciones?**  
En el E1, sí para `validar_ventas`. Mejor si también separas cargar / agregar / exportar.

**¿Qué hago con las filas inválidas?**  
No las borres en silencio: sepáralas, cuéntalas y (si aplica) expórtalas.

**¿`inplace=True` está mal?**  
No es “ilegal”, pero en notebooks de curso complica el razonamiento. Prefiere asignación.

**¿Por qué 8 y 2 en el checkpoint?**  
Porque el dataset del curso está **diseñado** con dos filas malas. Si no te salen, revisa las reglas `> 0` y `to_numeric`.

**¿Iris entra en la entrega?**  
No. Es material de práctica. El E1 usa `ventas.csv`.

---

## 20. Cierre

Si dominas este notebook, tienes la base de datos de DSIA:

> **Diagnosticar → validar con reglas → agregar → exportar, siempre con rutas portables.**

Eso no es burocracia: es lo que permite arquitectura limpia, tests y, más adelante, APIs e IA sobre datos fiables.

**Siguiente paso inmediato:** completa [`ejercicios/E1_pandas.md`](ejercicios/E1_pandas.md) en tu repo del Proyecto I.
